# AILLA — process data

Aggregates the parsed AILLA JSON into per-language resource sizes.

In [ ]:
import json
from itertools import chain

In [ ]:
def save_to_json_file(data, filename):

    # Convert the list of dictionaries to a JSON string
    json_string = json.dumps(data, indent=4, ensure_ascii=False)
    
    # Write the JSON string to a file
    with open(filename, 'w') as file:
        file.write(json_string)

def read_json_from_file(filename):

    with open(filename, 'r') as file:
        data = json.load(file)
    
    return data

### One-time construction of `ailla_entry_list.json`

The block below built `ailla_entry_list.json` from the raw `all_files.json`. It is kept for reference; subsequent cells simply load the saved file.

In [ ]:
# --- Reference: build ailla_entry_list.json from all_files.json (run once) ---

# all_entry_list = read_json_from_file("./all_files.json")

# print(len(all_entry_list))

# entry_list = []

# for entry in all_entry_list:
#     new_entry = {}
#     new_entry["id"] = entry["id"]
#     new_entry["filename"] = entry["filename"]
#     new_entry["language_id_list"] = entry["media_language"]
#     new_entry["language_detail_list"] = entry["media_language_detail"]
#     new_entry["type"] = entry["media_type"]
#     new_entry["place_created"] = entry["place_created"]
#     new_entry["date_uploaded"] = entry["date_uploaded"]
#     new_entry["file_size"] = entry["file_size"]
#     new_entry["height"] = entry["height"]
#     new_entry["width"] = entry["width"]
#     new_entry["extent"] = entry["extent"]
#     entry_list.append(new_entry)

# len(entry_list)

# save_to_json_file(entry_list, "ailla_entry_list.json")

# entry_list[1250]

In [ ]:
entry_list = list(filter(lambda x : x["type"], entry_list))

In [ ]:
len(entry_list)

In [ ]:
save_to_json_file(entry_list, "./ailla_entry_list.json")

In [ ]:
entry_list = read_json_from_file("./ailla_entry_list.json")

In [ ]:
language_detail_list = list(chain(*(map(lambda x : x["language_detail_list"], entry_list))))

In [ ]:
id_language_dic = {}

for item in language_detail_list:
    id_language_dic[item["id"]] = item

In [ ]:
lang_code_set = set()
lang_code_name_dic = {}



for lan_id, lan_details in id_language_dic.items():
    
    lang_code_set.add(lan_details["language_code"])
    lang_code_name_dic[lan_details["language_code"]] = lan_details["name"]["en"]

In [ ]:
save_to_json_file(lang_code_name_dic, "lang_code_name_dic.json")

In [ ]:
entry_list[0]

In [ ]:
for entry in entry_list:
    entry["language_code_list"] = list(map(lambda x : id_language_dic[x]["language_code"], entry["language_id_list"]))
    entry["language_name_list"] = list(map(lambda x : id_language_dic[x]["name"]["en"], entry["language_id_list"]))

In [ ]:
entry_list[0]

In [ ]:
id_language_dic

In [ ]:
len(language_detail_list)

In [ ]:
len(language_detail_list)

In [ ]:
def parse_time_to_seconds(time_str):
    hours, minutes, seconds = map(int, time_str.split(':'))
    total_seconds = hours * 3600 + minutes * 60 + seconds
    return total_seconds

In [ ]:
language_resource_dic = {}
temp_set = set()
con = 0
ok_con = 0
for entry in entry_list:
    langugage_code_list = entry["language_code_list"]
    for code in langugage_code_list:
        this_resource_dic = language_resource_dic.get(code, {})
        this_file_type = entry["type"]
        
        file_count_key = this_file_type + "_file_count"
        file_count = this_resource_dic.get(file_count_key, 0)
        file_count += 1
        this_resource_dic[file_count_key] = file_count
        
        
        
        size_key = this_file_type + "_size"
        file_size = this_resource_dic.get(size_key, 0)
        file_size = file_size + entry["file_size"]
        this_resource_dic[size_key] = file_size
        
        
        if this_file_type in ["audio", "video"]:
            length_key = this_file_type + "_length"
            length_size = this_resource_dic.get(length_key, 0)
            length_size = length_size + parse_time_to_seconds(entry["extent"])
            this_resource_dic[length_key] = length_size
        

        if this_file_type in ["image"]:
            pixel_key = this_file_type + "_pixel"
            pixel_size = this_resource_dic.get(pixel_key, 0)
            if not entry["height"]:
                con += 1
                continue
        
            pixel_size = pixel_size + (entry["height"] * entry["width"])
            this_resource_dic[pixel_key] = pixel_size
        
        
        
        
        language_resource_dic[code] = this_resource_dic
            
#         temp_set.add(this_file_type)

In [ ]:
save_to_json_file(language_resource_dic, "language_resource_dic.json")

In [ ]:
language_resource_dic

In [ ]:
entry_list[0]